# 06 · Hipótesis del equipo

**Qué responde:** cuáles de nuestras ideas se sostienen con los datos antes de entrar al reporte final.
Juntamos todo lo que hemos planteado, con la misma regla para todo:
- H1–H10 del EDA y del ADN del Entrenador (notebooks 04 y 05);
- el ADN del club y el "ADN ganador" (liguillas, finales, rivales);
- la revisión con ojos de entrenador (`Comentarios Emi.md`): hipótesis nuevas y forma de contarlo.

**Reglas**
- **Ventana oficial (decidida): Ap 2021 – Cl 2025**; Solari, Ortiz y Jardine. Lo de 2025–26 queda aparte, como epílogo marcado.
- Se compara **dentro del mismo técnico y del mismo nivel de rival** (`hipotesis.strat_perm`, permutación estratificada).
- El plan de juego (posesión, presión) se mide **con el partido empatado**: con el marcador movido, el plan es consecuencia del resultado.
- Veredicto con reglas fijas (`hipotesis.veredicto`): **se sostiene** si p < 0.10 y el efecto es de al menos 0.2 desviaciones estándar, con 8 partidos o más por grupo (5 series o torneos en los análisis por serie o torneo); **no cambia** si no; **no alcanza** si faltan partidos o el dato.
- Lo que no se sostiene también va al reporte: "lo probamos y no cambia el resultado".

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import cube as K, hipotesis as H, story as S
from src.pipeline.config import GOLD

pd.set_option("display.max_columns", 30, "display.width", 250, "display.float_format", lambda v: f"{v:,.2f}")
OUT = GOLD / "analysis"
CUBE = K.load(); P = CUBE["partidos"]; T = CUBE["tramos"]; EJES = CUBE["ejes"].eje.tolist()
W = H.ventana(P)
COLS = ["metrica", "si", "no", "dif", "dif_sd", "p", "n_si", "n_no", "veredicto"]
REG = []   # registro: una fila por hipótesis

def reg(id_, origen, hipotesis, evidencia, veredicto, seccion):
    REG.append(dict(id=id_, origen=origen, hipotesis=hipotesis, evidencia=evidencia, veredicto=veredicto, seccion=seccion))

def row(df, metrica):
    return df.set_index("metrica").loc[metrica]

def save(df, name):
    df.to_parquet(OUT / f"hipotesis_{name}.parquet", index=False)

print(W.coach.value_counts().to_dict())

{'Jardine': 91, 'Ortiz': 55, 'Solari': 27}


## A · ¿Qué es del club y qué es del técnico?

- del club son la solidez defensiva y el balón parado; el resto cambia con el técnico.
- el ADN del club es el control: balón, campo rival, ataque en control, con los tres técnicos.
- la posesión es el rasgo común de los técnicos.

Prueba: cada rasgo del ADN de 0 a 100 frente a la liga, por técnico y por torneo. Es "del club" si los tres técnicos y
todos los torneos quedan del mismo lado de 50.

In [2]:
adn = H.adn_comun(P, CUBE["adn_ref"], EJES)
display(adn.round(0))
save(adn.reset_index(names="rasgo"), "adn_comun")
club = adn[adn.del_club == "siempre arriba de la liga"]
txt = "; ".join(f"{r.lower()} ({adn.loc[r, 'torneo_min']:.0f}–{adn.loc[r, 'torneo_max']:.0f} de 100 según el torneo)" for r in club.index)
bp = adn.loc["Balón parado"]
reg("A1", "Equipo", "Hay un ADN del club que no depende del técnico",
    f"Con Solari, Ortiz y Jardine, y en todos los torneos, quedan arriba de la liga: {txt}. "
    f"El balón parado no: Solari {bp.Solari:.0f}, y un torneo en {bp.torneo_min:.0f}. Verticalidad, presión alta y reacción tras pérdida cambian con el técnico.",
    "se sostiene (corrige H2: el club es control, peligro y solidez; el balón parado depende del técnico)", "1 · El ADN del América")

,Solari,Ortiz,Jardine,torneo_min,torneo_max,del_club
Control del balón,68.00,93.00,90.00,53.00,100.00,siempre arriba de la liga
Verticalidad,19.00,65.00,49.00,10.00,71.00,depende del técnico
Llegada al área,31.00,96.00,95.00,21.00,100.00,depende del técnico
Presión alta,66.00,86.00,37.00,17.00,89.00,depende del técnico
Reacción tras pérdida,4.00,71.00,46.00,6.00,77.00,depende del técnico
Peligro generado,54.00,98.00,97.00,63.00,99.00,siempre arriba de la liga
Solidez defensiva,91.00,94.00,98.00,70.00,100.00,siempre arriba de la liga
Balón parado,62.00,79.00,74.00,47.00,89.00,depende del técnico


## B · Contexto: localía, plan de visita, presión y minuto

- la localía no distinguía. la localía cambia el resultado, no el estilo.
- de visita conviene jugar más defensivo. jugar pasivo o agresivo.
- ¿en qué momento del partido le generan más peligro?

In [3]:
loc = H.localia(P)
display(loc[COLS]); save(loc, "localia")
pts, dom, hp = row(loc, "Puntos por partido"), row(loc, "Dominio territorial (%)"), row(loc, "Acciones defensivas en campo rival (%)")
ppda = row(loc, "Pases que deja dar (PPDA)")
reg("B1", "Equipo", "La localía cambia el resultado pero no el estilo",
    f"En casa suma {pts.dif:+.2f} puntos por partido frente a visita (mismo técnico y mismo nivel de rival) "
    f"y también cambia el estilo: {dom.dif:+.0f} puntos de dominio territorial y {hp.dif:+.0f} de acciones defensivas en campo rival. "
    f"Lo que no cambia es cuánto deja jugar al rival antes de presionar ({ppda.dif:+.1f} pases).",
    "se sostiene en resultado y en estilo (corrige la nota: la localía sí distingue)", "3 · Cómo ajusta")

,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Puntos por partido,2.09,1.70,0.39,0.31,0.05,86,86,se sostiene
1,Ocasiones a favor (xG),1.51,1.32,0.23,0.29,0.05,86,86,se sostiene
2,Ocasiones en contra (xG),0.78,0.92,-0.15,-0.30,0.05,86,86,se sostiene
3,Posesión (%),54.72,52.26,2.65,0.30,0.04,86,86,se sostiene
4,Dominio territorial (%),62.68,55.90,7.09,0.47,0.00,86,86,se sostiene
5,Pases que deja dar (PPDA),14.42,13.70,0.68,0.09,0.54,86,86,no cambia
6,Acciones defensivas en campo rival (%),43.87,39.31,4.95,0.62,0.00,86,86,se sostiene
7,Toques en el área rival (por 100 posesiones),53.84,46.37,8.76,0.47,0.00,86,86,se sostiene


In [4]:
vis = H.plan_y_puntos(P, T, "possession_time_pct", "bajo", {"localia": "Visita"})
pas = H.plan_y_puntos(P, T, "ppda", "alto", {"fase": "Fase regular"})
display(vis[COLS], pas[COLS]); save(vis, "plan_visita"); save(pas, "plan_pasivo")
v, q = row(vis, "Puntos por partido"), row(pas, "Puntos por partido")
reg("B2", "Equipo", "De visita conviene ceder el balón y jugar más defensivo",
    f"De visita, los partidos en que tuvo menos balón con el marcador empatado sumaron {v.si:.2f} puntos frente a {v.no:.2f} "
    f"(diferencia dentro del mismo técnico y rival {v.dif:+.2f}, p = {v.p:.2f}); las ocasiones a favor y en contra tampoco cambian.",
    v.veredicto, "6 · Lo que probamos")
reg("B3", "Equipo", "Presionar menos (jugar pasivo) da más puntos en la fase regular",
    f"Con presión de espera (más pases al rival con el partido empatado) sumó {q.si:.2f} puntos frente a {q.no:.2f} "
    f"(dif. {q.dif:+.2f}, p = {q.p:.2f}). En la fase regular, el plan pasivo no cambia el resultado; en liguilla sí (ver C2).",
    q.veredicto, "6 · Lo que probamos")

,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Puntos por partido,1.73,1.59,0.18,0.15,0.55,30,36,no cambia
1,Ocasiones a favor (xG),1.25,1.28,-0.03,-0.04,0.86,30,36,no cambia
2,Ocasiones en contra (xG),0.90,0.87,0.01,0.03,0.91,30,36,no cambia
3,possession_time_pct con el partido empatado,43.69,57.47,-13.67,-1.47,0.00,30,36,se sostiene


,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Puntos por partido,2.11,1.85,0.25,0.21,0.27,47,52,no cambia
1,Ocasiones a favor (xG),1.44,1.43,0.01,0.01,0.94,47,52,no cambia
2,Ocasiones en contra (xG),0.86,0.81,0.04,0.09,0.65,47,52,no cambia
3,ppda con el partido empatado,22.91,9.92,12.90,1.35,0.00,47,52,se sostiene


In [5]:
seg = pd.read_parquet(GOLD / "fct_segment.parquet")
mins = {c: H.peligro_por_minuto(seg, W[W.coach == c].match_id).assign(tecnico=c) for c in ["Jardine"]}
mins["Todos"] = H.peligro_por_minuto(seg, W.match_id).assign(tecnico="Los tres técnicos")
mm = pd.concat(mins.values()); display(mm); save(mm, "minuto")
j = mins["Jardine"].set_index("tramo")
reg("B4", "Equipo", "Hay un momento del partido en que le generan más peligro",
    f"Con Jardine, el cierre del primer tiempo (30–45') es el tramo de más peligro rival: {j.loc['30-45+', 'xg_contra_90']:.2f} xG por 90' "
    f"(margen {j.loc['30-45+', 'xg_contra_90_lo']:.2f}–{j.loc['30-45+', 'xg_contra_90_hi']:.2f}), y el arranque del segundo (45–60') el de menos: "
    f"{j.loc['45-60', 'xg_contra_90']:.2f} ({j.loc['45-60', 'xg_contra_90_lo']:.2f}–{j.loc['45-60', 'xg_contra_90_hi']:.2f}). "
    f"Su ataque crece al final: {j.loc['75-90', 'xg_favor_90']:.2f} xG por 90' en 75–90' frente a {j.loc['0-15', 'xg_favor_90']:.2f} al inicio.",
    "se sostiene", "5 · Puntos críticos")

,tramo,minutos,xg_favor_90,xg_contra_90,xg_contra_90_lo,xg_contra_90_hi,tecnico
0,0-15,903.57,1.70,1.11,0.84,1.42,Jardine
1,15-30,828.78,1.81,1.12,0.80,1.50,Jardine
2,30-45+,863.53,1.93,1.41,1.16,1.68,Jardine
3,45-60,"1,517.42",1.54,0.89,0.72,1.08,Jardine
4,60-75,903.17,2.31,1.14,0.89,1.41,Jardine
5,75-90,929.90,2.35,1.11,0.85,1.38,Jardine
6,90+,359.10,2.40,1.48,0.96,2.09,Jardine
0,0-15,"1,638.38",1.60,1.14,0.92,1.36,Los tres técnicos
1,15-30,"1,535.37",1.94,1.26,1.03,1.53,Los tres técnicos
2,30-45+,"1,549.72",2.00,1.30,1.13,1.49,Los tres técnicos


## C · Ganar y perder

- el campeón produce más en la fase regular y en la liguilla cede el balón y presiona menos.
- en liguilla el equipo se vuelve reactivo sin conceder más.
- ¿por qué Ortiz perdió sus tres semifinales? jugar la vuelta en casa ayuda a remontar.
- ¿qué patrones hay cuando pierde?

In [6]:
PV = P[P.muestra == "ventana"]          # ventana decidida: Ap 2021 – Cl 2025
TT = K.torneos(PV); TT = TT[(TT.llego != "En curso") & TT.tecnico.isin(H.COACHES)].assign(campeon=lambda d: d.llego.eq("Campeón"))
reg_cols = {"xg_favor_regular": "Ocasiones por partido, fase regular (xG)", "posesion_regular": "Posesión, fase regular (%)",
            "puntos_regular": "Puntos por partido, fase regular"}
camp = H.compare(TT, reg_cols, "campeon", strata=[]).assign(veredicto=lambda d: d.apply(H.veredicto, axis=1, min_group=3))
display(TT, camp); save(camp, "campeon_regular")
c = row(camp, "Ocasiones por partido, fase regular (xG)")
reg("C1", "Equipo", "El campeón produce más ocasiones en la fase regular",
    f"Torneos campeones: {c.si:.2f} xG por partido en fase regular; el resto: {c.no:.2f}. Son {c.n_si} torneos ganados contra {c.n_no}: "
    f"la diferencia no es mayor que la que daría el azar (p = {c.p:.2f}). El Ap 2024 fue campeón con una fase regular discreta.",
    "no alcanza (pista, no regla)", "4 · Por qué ganó")

,torneo,tecnico,inicio,llego,partidos_regular,puntos_regular,xpts_regular,xg_favor_regular,posesion_regular,ppda_regular,partidos_liguilla,posesion_liguilla,ppda_liguilla,field_tilt_liguilla,campeon
0,Apertura 2021,Solari,2021-07-23,Cuartos,17,2.06,1.69,1.07,52.53,10.23,2,53.24,10.90,54.50,False
1,Clausura 2022,Solari,2022-01-08,Semifinal,8,0.75,1.44,1.02,48.85,9.91,0,NaN,NaN,NaN,False
2,Clausura 2022,Ortiz,2022-03-06,Semifinal,9,2.22,2.12,1.68,51.00,11.44,4,54.68,6.36,56.93,False
3,Apertura 2022,Ortiz,2022-07-03,Semifinal,17,2.24,1.85,1.48,54.50,12.16,4,60.71,8.24,73.53,False
4,Clausura 2023,Ortiz,2023-01-07,Semifinal,17,2.00,1.91,1.53,54.62,10.96,4,51.47,14.74,51.69,False
5,Apertura 2023,Jardine,2023-07-01,Campeón,17,2.35,2.12,1.79,55.02,13.76,6,51.15,18.76,60.63,True
6,Clausura 2024,Jardine,2024-01-14,Campeón,17,2.06,1.81,1.63,52.35,16.17,6,50.80,16.00,56.49,True
7,Apertura 2024,Jardine,2024-07-06,Campeón,17,1.59,1.79,1.42,56.62,12.13,7,42.41,26.64,41.23,True
9,Clausura 2025,Jardine,2025-01-26,Final perdida,15,2.00,1.83,1.60,56.44,12.00,6,56.31,11.47,65.11,False


,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,"Ocasiones por partido, fase regular (xG)",1.61,1.40,0.22,0.82,0.26,3,6,no cambia
1,"Posesión, fase regular (%)",54.66,52.99,1.67,0.65,0.38,3,6,no cambia
2,"Puntos por partido, fase regular",2.00,1.88,0.12,0.25,0.80,3,6,no cambia


In [7]:
s = H.series_liguilla(PV, CUBE["balon_parado"])
display(s.drop(columns=["torneo"])); save(s, "series_liguilla")
ser_cols = {"ppda": "Pases que deja dar antes de presionar (PPDA)", "posesion": "Posesión (%)", "field_tilt": "Dominio territorial (%)",
            "xg_favor": "Ocasiones a favor en la serie (xG)", "xg_contra": "Ocasiones en contra en la serie (xG)",
            "bp_xg_favor": "Balón parado a favor (xG)", "bp_xg_contra": "Balón parado en contra (xG)"}
ser = H.compare(s, ser_cols, "avanzo", strata=["tecnico"])      # mismo técnico
ser_all = H.compare(s, ser_cols, "avanzo", strata=[])          # todas las series juntas
display(ser, ser_all); save(ser, "series_avanzo"); save(ser_all, "series_avanzo_todas")
PP, PO, XC = "Pases que deja dar antes de presionar (PPDA)", "Posesión (%)", "Ocasiones en contra en la serie (xG)"
pp, po, xc, pa, poa = row(ser, PP), row(ser, PO), row(ser, XC), row(ser_all, PP), row(ser_all, PO)
reg("C2", "Equipo", "En liguilla, el que avanza cede el balón y presiona menos, sin conceder más",
    f"Series que pasó ({pa.n_si:.0f}) frente a las que perdió ({pa.n_no:.0f}): deja dar {pa.si:.1f} pases al rival antes de presionar frente a {pa.no:.1f} "
    f"(p = {pa.p:.2f}; con el mismo técnico, p = {pp.p:.2f} con {pp.n_no:.0f} series perdidas). La posesión baja ({poa.si:.0f} % frente a {poa.no:.0f} %; "
    f"p = {poa.p:.2f} con todas las series y {po.p:.2f} con el mismo técnico). Las ocasiones en contra no suben ({xc.si:.2f} frente a {xc.no:.2f} xG por serie).",
    ("se sostiene en la presión (pocas series)" + ("; en la posesión, solo pista" if po.p >= 0.10 else " y en la posesión"))
    if pa.p < 0.10 and pp.p < 0.10 else "no alcanza",
    "4 · Por qué ganó")
o = s[(s.tecnico == "Ortiz") & (s.ronda == "Semifinal")]
txt = "; ".join(f"{r.rival} {r['global']} (xG {r.xg_favor:.1f}–{r.xg_contra:.1f}, deja dar {r.ppda:.0f} pases)" for _, r in o.iterrows())
jw = s[(s.tecnico == "Jardine") & s.avanzo.astype(bool)]
reg("C3", "Equipo", "Ortiz perdió sus tres semifinales queriendo dominar",
    f"Semifinales de Ortiz: {txt}. En dos tuvo ocasiones iguales o mejores que el rival y perdió; presionó más que en cualquier serie que el América ganó "
    f"(las series que pasó Jardine: mediana de {jw.ppda.median():.0f} pases antes de presionar).",
    "se sostiene (descriptivo, 3 series)", "4 · Por qué ganó")
bpf, bpc = row(ser, "Balón parado a favor (xG)"), row(ser, "Balón parado en contra (xG)")
bfa = row(ser_all, "Balón parado a favor (xG)")
reg("C4", "Equipo", "El balón parado inclinó las liguillas ganadas",
    f"En las series que pasó generó {bpf.si:.2f} xG de balón parado por serie frente a {bpf.no:.2f} en las que perdió "
    f"(p = {bpf.p:.2f} con el mismo técnico; {bfa.p:.2f} con todas las series) y concedió parecido ({bpc.si:.2f} frente a {bpc.no:.2f}). "
    "Los goles de balón parado en liguilla falta recalcularlos con la ventana y nuestro marco.",
    ("se sostiene en ocasiones a favor (pocas series)" if bpf.p < 0.10 and bfa.p < 0.10 else "no cambia en ocasiones") + "; goles por recalcular",
    "4 · Por qué ganó")

,serie,tecnico,ronda,rival,nivel_rival,partidos,vuelta_en_casa,ida,global,avanzo,xg_favor,xg_contra,posesion,ppda,field_tilt,bp_xg_favor,bp_xg_contra,puntos_esperados,llego,merecia
0,Apertura 2021 · Cuartos,Solari,Cuartos,Pumas UNAM,Medio,2,True,0-0,1-3,False,0.96,1.76,53.24,10.90,54.50,0.41,0.99,1.22,Cuartos,False
1,Clausura 2022 · Cuartos,Ortiz,Cuartos,Puebla,Medio,2,True,1-1,4-3,True,3.11,1.73,53.96,13.35,54.11,0.50,1.53,1.84,Semifinal,True
2,Clausura 2022 · Semifinal,Ortiz,Semifinal,Pachuca,Fuerte,2,False,1-1,1-4,False,2.50,2.50,55.40,6.02,59.75,0.54,0.75,1.05,Semifinal,True
3,Apertura 2022 · Cuartos,Ortiz,Cuartos,Puebla,Medio,2,True,6-1,11-2,True,6.07,1.50,58.53,11.99,71.76,1.80,0.41,2.80,Semifinal,True
4,Apertura 2022 · Semifinal,Ortiz,Semifinal,Toluca,Fuerte,2,True,1-2,2-3,False,2.83,2.41,62.88,7.47,75.31,0.92,1.50,1.18,Semifinal,True
5,Clausura 2023 · Cuartos,Ortiz,Cuartos,Atlético San Luis,Medio,2,True,3-1,4-3,True,3.23,1.89,48.67,14.74,49.86,1.34,1.01,1.85,Semifinal,True
6,Clausura 2023 · Semifinal,Ortiz,Semifinal,Guadalajara,Fuerte,2,True,1-0,2-3,False,1.09,1.95,54.26,13.14,53.52,0.63,0.61,0.92,Semifinal,False
7,Apertura 2023 · Cuartos,Jardine,Cuartos,León,Medio,2,True,2-2,4-2,True,3.09,1.46,56.38,18.53,66.39,1.16,0.91,2.33,Campeón,True
8,Apertura 2023 · Semifinal,Jardine,Semifinal,Atlético San Luis,Medio,2,True,5-0,5-2,True,2.93,2.90,46.90,31.17,53.14,1.72,1.36,1.37,Campeón,True
9,Apertura 2023 · Final,Jardine,Final,Tigres UANL,Fuerte,2,True,1-1,4-1,True,2.52,1.21,50.17,14.41,62.35,1.43,0.97,2.26,Campeón,True


,metrica,si,no,dif,dif_sd,p,n_si,n_no
0,Pases que deja dar antes de presionar (PPDA),19.49,9.59,6.82,0.79,0.07,15,4
1,Posesión (%),50.26,56.98,-6.06,-0.84,0.14,15,4
2,Dominio territorial (%),55.53,62.55,-8.32,-0.65,0.28,15,4
3,Ocasiones a favor en la serie (xG),2.54,1.65,1.72,1.45,0.00,15,4
4,Ocasiones en contra en la serie (xG),1.68,1.98,-0.20,-0.34,0.53,15,4
5,Balón parado a favor (xG),1.04,0.54,0.62,1.51,0.03,15,4
6,Balón parado en contra (xG),0.79,0.85,0.15,0.33,0.63,15,4


,metrica,si,no,dif,dif_sd,p,n_si,n_no
0,Pases que deja dar antes de presionar (PPDA),19.49,9.59,9.90,1.16,0.02,15,5
1,Posesión (%),50.26,56.98,-6.72,-0.95,0.06,15,5
2,Dominio territorial (%),55.53,62.55,-7.01,-0.56,0.30,15,5
3,Ocasiones a favor en la serie (xG),2.54,1.65,0.88,0.74,0.16,15,5
4,Ocasiones en contra en la serie (xG),1.68,1.98,-0.29,-0.51,0.33,15,5
5,Balón parado a favor (xG),1.04,0.54,0.50,1.19,0.01,15,5
6,Balón parado en contra (xG),0.79,0.85,-0.06,-0.15,0.79,15,5


In [8]:
iv = H.ida_vuelta(PV); display(iv); save(iv, "ida_vuelta")
ct = pd.crosstab(s.vuelta_en_casa, s.avanzo); display(ct)
casa = s[s.vuelta_en_casa]; fuera = s[~s.vuelta_en_casa]
reg("C5", "Equipo", "Cerrar la serie en casa ayuda a remontar",
    f"El América cerró en casa {len(casa)} de {len(s)} series (por ser mejor sembrado) y pasó {casa.avanzo.sum():.0f}; de las {len(fuera)} que cerró fuera pasó "
    f"{fuera.avanzo.sum():.0f}, {fuera[fuera.torneo == 'Apertura 2024'].avanzo.sum():.0f} de ellas en el Ap 2024, que ganó cerrando siempre de visita. La vuelta en casa casi nunca cambia, así que su efecto no se puede separar "
    "de ser el mejor sembrado.", "no alcanza", "6 · Lo que probamos")

,pierna,localia,partidos,puntos,goles_favor,goles_contra,xg_favor,xg_contra,posesion
0,ida,Local,5,1.80,1.00,0.40,1.09,0.48,50.70
1,ida,Visita,14,1.43,1.57,0.79,1.33,0.90,52.96
2,vuelta,Local,14,1.86,1.71,1.14,1.27,0.90,53.78
3,vuelta,Visita,5,1.40,1.40,1.80,0.67,1.30,43.03


avanzo,False,True
vuelta_en_casa,,
False,2,4
True,3,11


In [9]:
der = H.derrotas(P, T); display(der[COLS]); save(der, "derrotas")
peores = H.peores_partidos(P); display(peores); save(peores.assign(match_date=peores.match_date.astype(str)), "peores_partidos")
q, xc, ct_, dom = (row(der, m) for m in ["Calidad de cada tiro (xG por tiro)", "Ocasiones en contra (xG)",
                                         "Peligro que concede al contragolpe", "Dominio territorial con el partido empatado (%)"])
pr = row(der, "Pases que deja dar con el partido empatado (PPDA)")
reg("C6", "Equipo", "Cuando pierde hay un patrón que se repite",
    f"En las derrotas (frente a las victorias, mismo técnico y nivel de rival): tira peor ({q.si:.2f} frente a {q.no:.2f} xG por tiro), "
    f"le hacen casi el doble de daño al contragolpe ({ct_.si:.2f} frente a {ct_.no:.2f}) y, con el partido empatado, domina menos el campo "
    f"({dom.si:.0f} % frente a {dom.no:.0f} %). La presión no cambia ({pr.si:.0f} frente a {pr.no:.0f} pases).",
    "se sostiene", "5 · Puntos críticos")

,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
2,Calidad de cada tiro (xG por tiro),0.07,0.12,-0.04,-0.92,0.00,31,93,se sostiene
0,Ocasiones a favor (xG),0.94,1.71,-0.70,-0.85,0.00,31,93,se sostiene
6,Peligro que concede al contragolpe,0.47,0.26,0.22,0.74,0.00,31,93,se sostiene
10,Dominio territorial con el partido empatado (%),51.91,63.61,-12.40,-0.70,0.02,17,53,se sostiene
3,Tiros (por 100 posesiones),11.05,14.10,-2.75,-0.58,0.00,31,93,se sostiene
1,Ocasiones en contra (xG),1.07,0.81,0.28,0.53,0.01,31,93,se sostiene
9,Posesión con el partido empatado (%),49.19,54.77,-5.44,-0.51,0.07,17,53,se sostiene
4,Toques en el área rival (por 100 posesiones),42.91,53.03,-8.57,-0.45,0.02,31,93,se sostiene
8,Peligro a balón parado a favor (xG),0.39,0.63,-0.21,-0.44,0.03,31,93,se sostiene
12,Pases largos con el partido empatado (%),18.12,16.49,1.50,0.29,0.22,17,53,no cambia


,match_date,torneo,coach,localia,opponent,rival_nivel,goals_for,goals_against,xg_favor,xg_contra,possession_time_pct,sistema,fase
21,2022-02-06,Clausura 2022,Solari,Local,Atlético San Luis,Débil,2.00,3.00,1.17,2.99,63.62,4-3-3,Fase regular
102,2023-12-10,Apertura 2023,Jardine,Local,Atlético San Luis,Medio,0.00,2.00,0.22,1.89,45.77,4-4-2,Liguilla
165,2025-04-05,Clausura 2025,Jardine,Visita,Pachuca,Medio,0.00,1.00,0.49,1.72,54.09,4-3-3,Fase regular
39,2022-05-23,Clausura 2022,Ortiz,Visita,Pachuca,Fuerte,0.00,3.00,1.28,2.05,56.30,4-2-3-1,Liguilla
137,2024-09-30,Apertura 2024,Jardine,Local,Pumas UNAM,Fuerte,0.00,1.00,0.51,1.09,60.73,3-4-2-1,Fase regular
144,2024-11-10,Apertura 2024,Jardine,Visita,Toluca,Fuerte,0.00,4.00,0.59,1.12,48.50,4-4-2,Fase regular
43,2022-07-24,Apertura 2022,Ortiz,Visita,Tijuana,Débil,0.00,2.00,0.09,0.60,64.42,4-2-3-1,Fase regular
111,2024-02-18,Clausura 2024,Jardine,Visita,Pachuca,Fuerte,1.00,2.00,0.89,1.40,53.03,4-2-3-1,Fase regular
18,2021-11-28,Apertura 2021,Solari,Local,Pumas UNAM,Medio,1.00,3.00,0.86,1.28,55.56,4-4-2,Liguilla
81,2023-05-22,Clausura 2023,Ortiz,Local,Guadalajara,Fuerte,1.00,3.00,0.37,0.70,54.31,4-2-3-1,Liguilla


## D · Sistema, plantilla y jugadores

- el dibujo cambia la defensa y el ataque (sus ejemplos: 4-4-2 más sólido, 3-4-3 más ofensivo; menos medios = menos agresivo). Se prueban **todas** las formaciones, no solo esas.
- el 4-2-3-1 es el sistema del club y el que mejor rinde; con tres centrales se genera menos y se concede más.
- había mejores jugadores cuando ganó. la pierna hábil (zurdo o diestro).
- meter o sacar jugadores cambia el partido. bandas y tiempo efectivo de juego.
- el dominio dejó de convertirse en ocasiones al final de la etapa de Jardine.

**Cómo se miden las formaciones.** No basta la titular: el técnico cambia de dibujo durante el partido. Cada evento y
cada posesión llevan la formación que había **en el campo** en ese momento (la titular y cada cambio táctico), la propia y
la del rival, y los indicadores del marco se calculan para esos minutos (`hipotesis.tramos_formacion`, con
`features.aggregate`). Cada formación se compara con el 4-2-3-1 **con el mismo técnico, el mismo nivel de rival y el
mismo marcador** (la formación para remontar no se compara con la de cerrar un partido), con intervalo del 90 %
remuestreando partidos. Ocasión clara = tiro sin penal con xG ≥ 0.30 (`framework.A["big_chance_xg"]`).

In [10]:
scope = S.match_scope(pd.read_parquet(GOLD / "dim_match.parquet"))
ev = S.load_story_events(scope)                      # en memoria; nunca se guarda
poss = S.load_possessions(scope)
TR = H.tramos_formacion(ev, poss).merge(W[["match_id", "coach", "rival_nivel", "localia", "fase", "result"]], on="match_id")
TR = H.familias(TR).assign(rival_atras=lambda d: d.formacion_rival.map(lambda f: f"Línea de {H.lineas(f)[0]}"))
foot = H.pierna_habil(ev)                            # para D4
pf = H.perfil_bandas(S.starting_lineups(ev), foot)
del ev, poss
save(TR, "tramos_formacion")
FORM = H.formaciones(TR, 200)
display(FORM); save(FORM.reset_index(), "formaciones")
print(f"{TR.minutos.sum():,.0f} minutos en {TR.match_id.nunique()} partidos")

,minutos,xg_f,xg_c,tiros_f,tiros_c,claras_f,claras_c,possession_time_pct,field_tilt,ppda,high_press_pct,box_touches_per100poss,opp_f3_entry_pct,partidos,técnicos,% minutos empatando,% minutos ganando,% minutos perdiendo,% minutos contra rivales fuertes
formacion,,,,,,,,,,,,,,,,,,,
4-2-3-1,"8,198.57",1.45,0.69,13.38,9.50,1.19,0.33,54.79,61.36,13.68,43.14,52.14,53.40,96,"Jardine 3783', Ortiz 3158', Solari 1258'",50.54,35.00,14.46,30.80
4-4-2,"4,816.75",1.30,0.73,12.39,9.88,0.95,0.43,54.03,58.98,16.78,41.16,53.74,54.54,71,"Jardine 2534', Ortiz 1723', Solari 560'",50.28,31.30,18.41,38.55
3-4-3,900.83,1.28,1.00,11.19,11.89,1.00,0.60,45.40,42.99,24.39,35.54,39.20,60.90,25,"Jardine 754', Solari 127', Ortiz 20'",47.90,42.69,9.41,60.21
4-3-3,827.73,1.02,1.08,12.18,10.44,0.65,0.65,56.48,61.84,12.50,42.79,42.81,53.64,11,"Jardine 470', Solari 200', Ortiz 158'",45.13,32.90,21.97,7.55
4-1-4-1,738.55,1.15,0.92,12.67,8.16,0.73,0.85,56.35,62.30,11.70,43.95,52.09,50.77,12,"Jardine 457', Solari 281'",51.15,45.92,2.94,47.76
3-4-2-1,495.02,0.73,0.83,7.82,10.36,0.55,0.91,45.59,43.83,23.12,33.85,34.79,62.65,10,"Jardine 465', Ortiz 18', Solari 12'",42.18,45.35,12.47,90.69
4-1-2-1-2,387.98,1.30,0.84,12.29,9.97,1.16,0.46,49.75,58.22,9.24,46.16,50.51,56.57,8,"Jardine 154', Ortiz 135', Solari 99'",47.32,0.66,52.02,15.49
3-5-2,357.30,0.75,0.99,8.82,12.85,0.50,0.50,44.18,45.13,16.13,41.57,37.83,54.85,10,"Jardine 324', Solari 34'",49.56,35.40,15.04,90.61
4-4-1-1,246.08,0.94,1.04,12.07,11.70,0.37,0.37,52.77,60.45,16.19,37.52,56.69,54.58,7,"Jardine 139', Ortiz 107'",84.99,4.79,10.22,23.47


17,256 minutos en 173 partidos


In [11]:
VS = pd.concat([H.comparar_formacion(TR, a) for a in FORM.index if a != "4-2-3-1"]
               + [H.comparar_formacion(TR.assign(atras=TR.atras.astype(str)), "Línea de 3", "Línea de 4", col="atras"),
                  H.comparar_formacion(TR.assign(delanteros=TR.delanteros.astype(str)), "2", "1", col="delanteros")])
save(VS, "formacion_vs")
show = VS[VS.veredicto != "no alcanza"].pivot_table(index="metrica", columns="formacion", values="dif").round(2)
display(show)
display(VS[VS.veredicto == "se sostiene"][["formacion", "frente_a", "metrica", "dif", "lo", "hi", "minutos_comparables"]])

formacion,2,3-4-2-1,3-4-3,4-1-4-1,4-3-3,4-4-2,Línea de 3
metrica,,,,,,,
Acciones defensivas en campo rival (%),-0.91,-8.06,-6.83,0.91,-0.84,-1.89,-4.57
Dominio territorial (%),-3.07,-10.32,-13.40,6.08,-2.00,-2.21,-12.17
Jugadas rivales que llegan a su último tercio (%),0.58,3.05,4.91,-5.05,1.08,0.75,3.80
Ocasiones a favor (xG por 90'),-0.04,-0.48,0.01,-0.10,-0.54,-0.05,-0.16
Ocasiones claras a favor por 90',-0.10,-0.49,-0.09,-0.34,-0.69,-0.20,-0.32
Ocasiones claras en contra por 90',0.04,0.39,0.24,0.69,0.49,0.08,0.13
Ocasiones en contra (xG por 90'),0.04,0.05,0.34,0.28,0.41,0.03,0.17
Pases que deja dar (PPDA),2.01,5.84,8.72,-3.02,-0.50,3.05,3.56
Posesión (%),-1.68,-5.52,-6.96,5.76,0.63,-0.83,-6.37


,formacion,frente_a,metrica,dif,lo,hi,minutos_comparables
8,4-4-2,4-2-3-1,Pases que deja dar (PPDA),3.05,0.66,5.28,"4,728.97"
6,3-4-3,4-2-3-1,Posesión (%),-6.96,-11.10,-2.61,862.07
7,3-4-3,4-2-3-1,Dominio territorial (%),-13.40,-21.62,-4.98,862.07
8,3-4-3,4-2-3-1,Pases que deja dar (PPDA),8.72,1.89,15.60,862.07
9,3-4-3,4-2-3-1,Acciones defensivas en campo rival (%),-6.83,-12.52,-1.46,862.07
0,4-3-3,4-2-3-1,Ocasiones a favor (xG por 90'),-0.54,-1.07,-0.13,822.27
4,4-3-3,4-2-3-1,Ocasiones claras a favor por 90',-0.69,-1.62,-0.06,822.27
10,4-3-3,4-2-3-1,Toques en el área rival (por 100 posesiones),-10.67,-22.63,-0.60,822.27
8,3-4-2-1,4-2-3-1,Pases que deja dar (PPDA),5.84,0.24,12.23,464.85
9,3-4-2-1,4-2-3-1,Acciones defensivas en campo rival (%),-8.06,-15.60,-0.41,464.85


In [12]:
def v(a, m, b="4-2-3-1"):
    return VS[(VS.formacion == a) & (VS.frente_a == b) & (VS.metrica == m)].iloc[0]

XGF, XGC = "Ocasiones a favor (xG por 90')", "Ocasiones en contra (xG por 90')"
CLF, TC = "Ocasiones claras a favor por 90'", "Tiros en contra por 90'"
POS, DOM, BOX, PPDA = "Posesión (%)", "Dominio territorial (%)", "Toques en el área rival (por 100 posesiones)", "Pases que deja dar (PPDA)"
f = FORM
reg("D1", "Equipo", "El dibujo cambia la defensa y el ataque (p. ej. el 4-4-2 defiende mejor que el 4-2-3-1)",
    f"El 4-2-3-1 ({f.loc['4-2-3-1', 'minutos']:,.0f} minutos) y el 4-4-2 ({f.loc['4-4-2', 'minutos']:,.0f}) son los dibujos del club con los tres técnicos. "
    f"Con el mismo técnico, rival y marcador rinden igual: ocasiones a favor {v('4-4-2', XGF).dif:+.2f} y en contra {v('4-4-2', XGC).dif:+.2f} xG por 90' "
    f"(márgenes que cruzan el cero). Lo único que cambia es que el 4-4-2 presiona menos ({v('4-4-2', PPDA).dif:+.1f} pases antes de presionar).",
    "no cambia (el 4-4-2 no defiende mejor; rinde igual que el 4-2-3-1)", "6 · Lo que probamos")

uso = TR[TR.atras == "Línea de 3"]
L3 = lambda m: v("Línea de 3", m, "Línea de 4")
reg("D1b", "Equipo", "Con tres centrales el equipo es más ofensivo (o más sólido)",
    f"Con línea de 3 (3-4-3, 3-4-2-1, 3-5-2; {uso.minutos.sum():,.0f} minutos en {uso.match_id.nunique()} partidos, casi todos de Jardine, "
    f"{uso[uso.rival_nivel == 'Fuerte'].minutos.sum() / uso.minutos.sum() * 100:.0f} % contra rivales fuertes), frente a línea de 4 con el mismo técnico, rival y marcador: "
    f"cede el balón ({L3(POS).dif:+.0f} puntos de posesión) y el campo ({L3(DOM).dif:+.0f} de dominio), "
    f"pisa menos el área ({L3(BOX).dif:+.0f} toques por 100 posesiones) y concede más tiros ({L3(TC).dif:+.1f} por 90'). "
    "No es más ofensivo ni más sólido: el equipo se repliega.",
    "se sostiene: se repliega sin protegerse más", "5 · Puntos críticos")

reg("D1c", "Equipo", "Con tres delanteros (4-3-3) ataca más",
    f"Al revés: frente al 4-2-3-1 genera menos ({v('4-3-3', XGF).dif:+.2f} xG y {v('4-3-3', CLF).dif:+.2f} ocasiones claras por 90'; "
    f"{f.loc['4-3-3', 'minutos']:,.0f} minutos). Con uno o dos delanteros no cambia nada ({v('2', XGF, '1').dif:+.2f} xG por 90').",
    "no se sostiene (genera menos)", "6 · Lo que probamos")

In [13]:
esp = pd.concat([H.comparar_formacion(TR[TR.rival_atras == r].assign(atras=lambda d: d.atras.astype(str)), "Línea de 3", "Línea de 4",
                                      col="atras").assign(rival=r) for r in ["Línea de 3", "Línea de 4"]])
fue = pd.concat([H.comparar_formacion(TR[TR.rival_nivel == "Fuerte"], "4-4-2", "4-2-3-1", strata=("coach", "game_state3")),
                 H.comparar_formacion(TR[TR.rival_nivel == "Fuerte"], "Línea de 3", "Línea de 4", col="atras",
                                      strata=("coach", "game_state3"))])
display(esp, fue); save(esp, "formacion_espejo"); save(fue, "formacion_fuertes")
e3 = esp[(esp.rival == "Línea de 3") & (esp.metrica == TC)].iloc[0]
reg("D1d", "Equipo", "Tres centrales funcionan contra un rival que también juega con tres (espejo)",
    f"Contra rivales con línea de 3, nuestra línea de 3 también concede más tiros que nuestra línea de 4 ({e3.dif:+.1f} por 90', margen "
    f"{e3.lo:+.1f} a {e3.hi:+.1f}) y pisa menos el área. El espejo no ayuda.", "no se sostiene", "6 · Lo que probamos")
g = fue[fue.formacion == "4-4-2"].set_index("metrica")
h = fue[fue.formacion == "Línea de 3"].set_index("metrica")
reg("D1e", "Equipo", "Hay una mejor alineación contra los rivales más fuertes",
    f"Contra rivales fuertes (mismo técnico y marcador): el 4-4-2 llega más al área que el 4-2-3-1 ({g.loc[BOX, 'dif']:+.0f} toques por 100 posesiones) "
    f"con la misma solidez ({g.loc[XGC, 'dif']:+.2f} xG en contra por 90'). Tres centrales cede el balón ({h.loc[POS, 'dif']:+.0f} puntos de posesión) "
    f"sin conceder menos ({h.loc[XGC, 'dif']:+.2f} xG en contra por 90').",
    "se sostiene (el 4-4-2 da más llegadas sin conceder más)", "7 · Recomendaciones")

,formacion,frente_a,metrica,dif,lo,hi,minutos_comparables,veredicto,rival
0,Línea de 3,Línea de 4,Ocasiones a favor (xG por 90'),-0.33,-0.97,0.09,971.25,no cambia,Línea de 3
1,Línea de 3,Línea de 4,Ocasiones en contra (xG por 90'),0.17,-0.21,0.51,971.25,no cambia,Línea de 3
2,Línea de 3,Línea de 4,Tiros a favor por 90',-2.50,-5.64,-0.32,971.25,se sostiene,Línea de 3
3,Línea de 3,Línea de 4,Tiros en contra por 90',3.82,1.83,6.69,971.25,se sostiene,Línea de 3
4,Línea de 3,Línea de 4,Ocasiones claras a favor por 90',-0.37,-1.13,0.29,971.25,no cambia,Línea de 3
5,Línea de 3,Línea de 4,Ocasiones claras en contra por 90',0.14,-0.36,0.76,971.25,no cambia,Línea de 3
6,Línea de 3,Línea de 4,Posesión (%),-5.58,-13.25,0.50,971.25,no cambia,Línea de 3
7,Línea de 3,Línea de 4,Dominio territorial (%),-16.16,-25.66,-7.43,971.25,se sostiene,Línea de 3
8,Línea de 3,Línea de 4,Pases que deja dar (PPDA),3.71,-2.98,12.24,971.25,no cambia,Línea de 3
9,Línea de 3,Línea de 4,Acciones defensivas en campo rival (%),-4.54,-10.71,1.23,971.25,no cambia,Línea de 3


,formacion,frente_a,metrica,dif,lo,hi,minutos_comparables,veredicto
0,4-4-2,4-2-3-1,Ocasiones a favor (xG por 90'),-0.02,-0.38,0.22,"1,856.68",no cambia
1,4-4-2,4-2-3-1,Ocasiones en contra (xG por 90'),0.07,-0.22,0.30,"1,856.68",no cambia
2,4-4-2,4-2-3-1,Tiros a favor por 90',0.02,-2.39,1.75,"1,856.68",no cambia
3,4-4-2,4-2-3-1,Tiros en contra por 90',-0.22,-3.40,2.37,"1,856.68",no cambia
4,4-4-2,4-2-3-1,Ocasiones claras a favor por 90',0.05,-0.51,0.46,"1,856.68",no cambia
5,4-4-2,4-2-3-1,Ocasiones claras en contra por 90',0.06,-0.37,0.41,"1,856.68",no cambia
6,4-4-2,4-2-3-1,Posesión (%),0.39,-4.10,4.82,"1,856.68",no cambia
7,4-4-2,4-2-3-1,Dominio territorial (%),1.92,-4.39,8.18,"1,856.68",no cambia
8,4-4-2,4-2-3-1,Pases que deja dar (PPDA),3.14,-1.46,8.56,"1,856.68",no cambia
9,4-4-2,4-2-3-1,Acciones defensivas en campo rival (%),0.17,-4.12,4.20,"1,856.68",no cambia


In [14]:
med = H.medios(P); display(med[COLS]); save(med, "medios")
pr, xf = row(med, "Pases que deja dar (PPDA)"), row(med, "Ocasiones a favor (xG)")
reg("D2", "Equipo", "Con menos medios defiende con menos agresividad y ataca más",
    f"Con 4 medios o menos (titular) presiona menos (deja dar {pr.si:.1f} pases frente a {pr.no:.1f}), pero no ataca más "
    f"({xf.si:.2f} frente a {xf.no:.2f} xG) ni concede más.", "se sostiene en la presión; no en el ataque", "6 · Lo que probamos")

,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Puntos por partido,1.82,1.94,-0.09,-0.07,0.65,67,105,no cambia
1,Ocasiones a favor (xG),1.27,1.51,-0.18,-0.23,0.11,67,105,no cambia
2,Ocasiones en contra (xG),0.91,0.81,0.11,0.23,0.13,67,105,no cambia
3,Toques en el área rival (por 100 posesiones),48.60,51.02,-1.64,-0.09,0.55,67,105,no cambia
4,Jugadas rivales que llegan a su último tercio (%),55.41,53.89,0.80,0.09,0.55,67,105,no cambia
5,Posesión (%),53.25,53.63,0.30,0.03,0.82,67,105,no cambia
6,Pases que deja dar (PPDA),15.75,12.99,2.22,0.30,0.03,67,105,se sostiene


In [15]:
pl = H.plantilla(CUBE["jugadores"], P); display(pl); save(pl, "plantilla")
rho = H.spearman(pl, "aporte90_once")
o, j = pl[pl.tecnico == "Ortiz"], pl[(pl.tecnico == "Jardine") & (pl.llego == "Campeón")]
reg("D3", "Equipo", "Cuando ganó tenía mejores jugadores",
    f"El aporte por 90' (OBV) del once más usado en las semifinales perdidas de Ortiz ({o.aporte90_once.min():.2f}–{o.aporte90_once.max():.2f}) "
    f"es igual o mayor que en los torneos campeones de Jardine ({j.aporte90_once.min():.2f}–{j.aporte90_once.max():.2f}). "
    f"Relación entre el aporte del once y hasta dónde llegó, en {len(pl)} torneos: {rho:.2f} (débil). "
    "Límite: el aporte depende también del sistema; no hay datos físicos (cuánto corre, ritmo).",
    "no cambia", "6 · Lo que probamos")

,tecnico,torneo,llego,aporte90_Ataque,aporte90_Defensa,aporte90_Medio,aporte90_Portero,aporte90_once,minutos_del_once_pct,avance
2,Jardine,Apertura 2023,Campeón,0.27,0.19,0.25,0.19,0.21,67.71,5
3,Jardine,Apertura 2024,Campeón,0.21,0.18,0.18,0.18,0.20,65.71,5
4,Jardine,Apertura 2025,Cuartos,0.25,0.18,0.17,0.16,0.21,71.53,2
5,Jardine,Clausura 2024,Campeón,0.28,0.17,0.23,0.32,0.24,72.18,5
6,Jardine,Clausura 2025,Final perdida,0.24,0.19,0.22,0.29,0.23,72.49,4
7,Jardine,Clausura 2026,Cuartos,0.17,0.16,0.18,0.26,0.18,63.75,2
8,Ortiz,Apertura 2022,Semifinal,0.23,0.18,0.31,0.18,0.24,76.50,3
9,Ortiz,Clausura 2022,Semifinal,0.25,0.22,0.23,0.23,0.24,82.10,3
10,Ortiz,Clausura 2023,Semifinal,0.25,0.20,0.27,0.31,0.25,70.27,3
11,Solari,Apertura 2021,Cuartos,0.22,0.21,0.14,0.37,0.21,66.35,2


In [16]:
por_tec = W.join(pf, on="match_id").groupby("coach")[["extremos_invertidos", "laterales_invertidos"]].mean().reindex(H.COACHES)
bpie = H.bandas_pierna(P, pf)
display(foot.pierna.value_counts(), por_tec, bpie[COLS])
save(por_tec.reset_index(), "pierna_por_tecnico"); save(bpie, "pierna_bandas")
bt = row(bpie, "Toques en el área rival (por 100 posesiones)"); bx = row(bpie, "Ocasiones a favor (xG)")
reg("D4", "Equipo", "La pierna hábil de los extremos cambia el ataque",
    f"Extremos a pierna cambiada (diestro por la izquierda o zurdo por la derecha) por partido: Solari {por_tec.loc['Solari', 'extremos_invertidos']:.1f}, "
    f"Ortiz {por_tec.loc['Ortiz', 'extremos_invertidos']:.1f}, Jardine {por_tec.loc['Jardine', 'extremos_invertidos']:.1f} (de 2). "
    f"Con los dos a pierna cambiada pisa más el área ({bt.dif:+.0f} toques por 100 posesiones, mismo técnico y nivel de rival; p = {bt.p:.2f}), "
    f"pero no genera más ocasiones ({bx.dif:+.2f} xG por partido, p = {bx.p:.2f}). Los laterales casi siempre juegan a pierna natural.",
    "se sostiene en llegadas al área; no en ocasiones", "2 · Lo que agregó Jardine")

pierna
Diestro        32
Zurdo          11
Ambidiestro     2
Name: count, dtype: int64

,extremos_invertidos,laterales_invertidos
coach,,
Solari,0.28,0.07
Ortiz,1.94,0.04
Jardine,1.68,0.03


,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Puntos por partido,2.03,1.88,-0.02,-0.02,0.95,95,19,no cambia
1,Ocasiones a favor (xG),1.56,1.25,0.12,0.14,0.56,95,19,no cambia
2,Calidad de cada tiro (xG por tiro),0.11,0.10,0.00,0.02,0.94,95,19,no cambia
3,Toques en el área rival (por 100 posesiones),55.49,42.64,10.46,0.56,0.03,95,19,se sostiene
4,Peligro que nace de centros (%),22.53,24.89,5.95,0.35,0.18,95,19,no cambia
5,Avanza por el centro (%),43.27,41.32,-1.59,-0.17,0.52,95,19,no cambia


In [17]:
subs = pd.read_parquet(GOLD / "fct_substitution.parquet")
cam = H.efecto_cambios(subs, P); display(cam); save(cam, "cambios")
pe = cam[(cam.marcador == "Perdiendo") & (cam.sale == "Sale un defensor")].iloc[0]
ga = cam[(cam.marcador == "Ganando") & (cam.sale == "Sale un atacante")].iloc[0]
reg("D5", "Equipo", "Meter o sacar jugadores cambia el partido",
    f"Perdiendo, cuando sale un defensor (entra alguien más ofensivo) las ocasiones a favor suben {pe.cambio_xg_favor:+.2f} xG por 100 posesiones "
    f"(margen {pe.cambio_xg_favor_lo:+.2f} a {pe.cambio_xg_favor_hi:+.2f}) y el dominio {pe.cambio_dominio:+.0f} puntos, sin que suban las del rival "
    f"({pe.cambio_xg_contra:+.2f}). Ganando, sacar a un atacante baja el dominio {ga.cambio_dominio:+.0f} puntos. "
    "Descriptivo: no separa el cambio de lo que el partido ya venía haciendo.",
    "se sostiene (descriptivo)", "3 · Cómo ajusta")

,marcador,sale,ventanas,cambio_xg_favor,cambio_xg_favor_lo,cambio_xg_favor_hi,cambio_xg_contra,cambio_xg_contra_lo,cambio_xg_contra_hi,cambio_dominio,cambio_dominio_lo,cambio_dominio_hi
0,Empatando,Mixto o mismo puesto,20,0.86,0.34,1.42,-0.24,-0.58,0.04,3.07,-10.19,15.30
1,Empatando,Sale un atacante,92,-0.44,-0.87,-0.00,0.06,-0.18,0.28,2.37,-1.29,5.83
2,Empatando,Sale un defensor,53,0.01,-0.66,0.66,0.01,-0.22,0.27,-2.69,-8.24,2.68
3,Ganando,Mixto o mismo puesto,47,0.22,-0.78,1.49,0.05,-0.30,0.41,-1.51,-8.57,5.72
4,Ganando,Sale un atacante,106,-0.47,-0.95,0.02,0.03,-0.27,0.32,-5.61,-9.91,-1.19
5,Ganando,Sale un defensor,64,-0.38,-1.05,0.29,0.10,-0.23,0.47,0.53,-5.46,6.67
6,Perdiendo,Mixto o mismo puesto,22,0.51,0.04,1.01,-0.99,-2.23,-0.06,4.90,-4.62,13.47
7,Perdiendo,Sale un atacante,29,0.61,-0.23,1.55,-0.66,-1.27,-0.03,4.09,-2.69,10.84
8,Perdiendo,Sale un defensor,41,0.80,0.33,1.31,-0.31,-1.58,0.80,8.77,1.07,16.53


In [18]:
bd = H.bandas(P); display(bd[COLS]); save(bd, "bandas")
poss = pd.read_parquet(GOLD / "fct_possession.parquet")
te = W.join(H.tiempo_efectivo(poss).rename("tiempo"), on="match_id")
tw = H.compare(te[te.result.isin(["W", "L"])].assign(perdio=lambda d: d.result.eq("L")), {"tiempo": "Minutos con balón en juego"}, "perdio")
display(te.groupby("coach").tiempo.mean(), tw); save(tw, "tiempo_efectivo")
t = tw.iloc[0]
reg("D6", "Equipo", "Ataca por otra banda cuando pierde", "El carril por el que avanza (izquierda, centro, derecha) es el mismo en derrotas y victorias. "
    "Por dónde le hacen daño no se puede medir con el cubo (casi todos los tiros son por el centro; haría falta el carril de entrada del rival).",
    "no cambia (ataque); no alcanza (defensa)", "6 · Lo que probamos")
reg("D7", "Equipo", "El tiempo efectivo de juego cambia el resultado",
    f"Minutos con balón en juego: {t.si:.0f} en derrotas frente a {t.no:.0f} en victorias (p = {t.p:.2f}).", H.veredicto(t), "6 · Lo que probamos")

,metrica,si,no,dif,dif_sd,p,n_si,n_no,veredicto
0,Avanza por la izquierda (%),30.42,31.14,-0.86,-0.10,0.66,31,93,no cambia
1,Avanza por el centro (%),40.90,42.14,-0.61,-0.06,0.77,31,93,no cambia
2,Avanza por la derecha (%),28.68,26.72,1.46,0.15,0.47,31,93,no cambia


coach
Jardine   69.29
Ortiz     63.50
Solari    59.88
Name: tiempo, dtype: float64

,metrica,si,no,dif,dif_sd,p,n_si,n_no
0,Minutos con balón en juego,66.25,65.64,1.26,0.15,0.47,31,93


In [19]:
jt = K.torneos(P); jt = jt[jt.tecnico == "Jardine"][["torneo", "llego", "xg_favor_regular", "posesion_regular"]]
display(jt); save(jt, "jardine_torneos")
reg("D8", "Equipo", "Al final de la etapa de Jardine el dominio dejó de convertirse en ocasiones",
    "Con la misma posesión en fase regular (" + ", ".join(f"{r.posesion_regular:.0f} %" for r in jt.itertuples()) + "), las ocasiones por partido bajan de "
    f"{jt.xg_favor_regular.iloc[0]:.2f} (Ap 2023) a {jt.xg_favor_regular.iloc[-1]:.2f} (Cl 2026). Depende de Ap 2025 y Cl 2026, fuera de la ventana: va como epílogo.",
    "se sostiene (descriptivo, epílogo)", "Epílogo · 2025–26")

,torneo,llego,xg_favor_regular,posesion_regular
5,Apertura 2023,Campeón,1.79,55.02
6,Clausura 2024,Campeón,1.63,52.35
7,Apertura 2024,Campeón,1.42,56.62
9,Clausura 2025,Final perdida,1.60,56.44
10,Apertura 2025,Cuartos,1.18,57.31
11,Clausura 2026,Cuartos,0.97,56.35


## E · Hipótesis ya cerradas en los notebooks 04 y 05

Se registran con la evidencia que ya tenían, para que el reporte tenga una sola tabla.

In [20]:
h4 = pd.read_parquet(OUT / "coach_hypotheses.parquet")
for k, lab, sec in [("H1", "se sostiene", "8 · Nuestro aporte"), ("H3", "se sostiene (leve)", "3 · Cómo ajusta"),
                    ("H9", "se sostiene", "8 · Nuestro aporte"), ("H10", "se sostiene", "8 · Nuestro aporte")]:
    reg(k, "Equipo", h4.loc[k, "hipótesis"], h4.loc[k, "evidencia"], lab, sec)
reg("H4", "Equipo", "El marcador cambia el plan", "Mismo partido frente al empate: perdiendo por 1, +8 de dominio territorial y +11 de posesión; "
    "ganando por 1, el dominio baja 17 puntos (guarda el balón lejos del área). Lo confirmamos con una segunda medición (minutos por estado del marcador).", "se sostiene", "3 · Cómo ajusta")
reg("H7", "Equipo", "El juego está repartido y pasa por el doble pivote", "Los 3 que más aportan generan el 62 % del valor con balón; "
    "Fidalgo es el que más aporta (11 %) y el que más participa en las jugadas de gol.", "se sostiene", "2 · Lo que agregó Jardine")
reg("H8", "Equipo", "El balón parado es arma y punto débil", "Da el 37 % de las ocasiones de Jardine (la mayoría en la segunda jugada); "
    "en contra, el primer palo concentra el 53 % del peligro de córner; ahí va la mayor parte de los córners rivales.", "se sostiene", "5 · Puntos críticos")
reg("L1", "Equipo", "Lesiones, cuánto corre, ritmo físico", "StatsBomb no trae datos físicos ni de lesiones.", "no alcanza (no hay dato)", "Anexo · Límites")

In [21]:
R = pd.DataFrame(REG)
R.to_parquet(OUT / "hipotesis_equipo.parquet", index=False)
pd.set_option("display.max_colwidth", 400)
display(R[["id", "origen", "hipotesis", "veredicto", "seccion"]])
display(R.groupby("veredicto").size())

,id,origen,hipotesis,veredicto,seccion
0,A1,Equipo,Hay un ADN del club que no depende del técnico,"se sostiene (corrige H2: el club es control, peligro y solidez; el balón parado depende del técnico)",1 · El ADN del América
1,B1,Equipo,La localía cambia el resultado pero no el estilo,se sostiene en resultado y en estilo (corrige la nota: la localía sí distingue),3 · Cómo ajusta
2,B2,Equipo,De visita conviene ceder el balón y jugar más defensivo,no cambia,6 · Lo que probamos
3,B3,Equipo,Presionar menos (jugar pasivo) da más puntos en la fase regular,no cambia,6 · Lo que probamos
4,B4,Equipo,Hay un momento del partido en que le generan más peligro,se sostiene,5 · Puntos críticos
5,C1,Equipo,El campeón produce más ocasiones en la fase regular,"no alcanza (pista, no regla)",4 · Por qué ganó
6,C2,Equipo,"En liguilla, el que avanza cede el balón y presiona menos, sin conceder más","se sostiene en la presión (pocas series); en la posesión, solo pista",4 · Por qué ganó
7,C3,Equipo,Ortiz perdió sus tres semifinales queriendo dominar,"se sostiene (descriptivo, 3 series)",4 · Por qué ganó
8,C4,Equipo,El balón parado inclinó las liguillas ganadas,se sostiene en ocasiones a favor (pocas series); goles por recalcular,4 · Por qué ganó
9,C5,Equipo,Cerrar la serie en casa ayuda a remontar,no alcanza,6 · Lo que probamos


veredicto
no alcanza                                                                                              1
no alcanza (no hay dato)                                                                                1
no alcanza (pista, no regla)                                                                            1
no cambia                                                                                               4
no cambia (ataque); no alcanza (defensa)                                                                1
no cambia (el 4-4-2 no defiende mejor; rinde igual que el 4-2-3-1)                                      1
no se sostiene                                                                                          1
no se sostiene (genera menos)                                                                           1
se sostiene                                                                                             8
se sostiene (corrige H2: el club es 